# 03 — LightGBM next-basket recommender

A supervised alternative to the same-basket association rules in notebook 02.
For each candidate product, LightGBM learns whether it appears in the user's next order.
We use the LightGBM library; A-Priori and PCY remain our own implementations.

1. A hand-checkable feature example
2. The same saved train / validation / test users as notebooks 01–02
3. Prior-order features and candidate coverage
4. Validation tuning, early stopping and feature importance
5. One final test comparison with popularity and the saved rule recommendations

Run notebook 01 first to save `split.csv`. The six raw CSVs are required (see README).
The code below has no saved full-data results yet. Runtime and memory depend on the
number of user-product candidates; test users are processed in batches.

In [ ]:
import sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), Path.cwd() / "project", Path.cwd().parent)
            if (p / "src" / "data.py").is_file())
sys.path.insert(0, str(ROOT))
from src.data import load_tables, next_basket_truth
from src.lightgbm_recommender import (
    FEATURES, product_features, make_features, candidate_labels,
    fit_lightgbm, recommend_lightgbm,
)
from src.recommend import evaluate_at_k

DERIVED = ROOT / "data" / "derived"
K, N_POPULAR = 10, 20

## 1. Check the features by hand

User 1 bought {1, 2}, then {1, 3}. Product 1 has count 2, purchase rate 1,
and recency 0; product 2 has count 1, rate 0.5, and recency 1.
The next order {1, 4} supplies labels only. Product 4 must not be inserted just
because it is in the target: it lies outside these candidates and still counts
in the denominator of recall.

In [ ]:
toy_orders = pd.DataFrame({
    "order_id": [11, 12, 13], "user_id": [1, 1, 1],
    "eval_set": ["prior", "prior", "train"], "order_number": [1, 2, 3],
})
toy_prior = pd.DataFrame({"order_id": [11, 11, 12, 12], "product_id": [1, 2, 1, 3]})
toy_products = pd.DataFrame({"product_id": [1, 2, 3, 4], "aisle_id": [1, 1, 2, 2],
                             "department_id": [1, 1, 1, 1]})
toy_stats = product_features(toy_orders, toy_prior, [1])
toy = make_features(toy_orders, toy_prior, toy_products, [1], toy_stats, n_popular=1)
f = toy.set_index("product_id")
assert f.loc[1, "purchase_count"] == 2 and f.loc[1, "purchase_rate"] == 1
assert f.loc[2, "purchase_rate"] == 0.5 and f.loc[2, "orders_since_last"] == 1
assert candidate_labels(toy, {1: {1, 4}}).tolist() == [1, 0, 0]
assert 4 not in f.index
print("Toy feature and label checks passed")
toy

## 2. Load the shared user split

Only `prior` orders enter the feature builder. Product popularity statistics are
fitted on the **training users' prior orders**, then frozen for validation and test.
Each evaluation user's own prior history is available at prediction time.
No target contents or target-order timing are used as inputs. User IDs and product IDs
are join keys, not model features; aisle and department IDs are categorical features.

In [ ]:
t = time.time()
tables = load_tables()
orders, prior, products = tables["orders"], tables["prior"], tables["products"]
truth = next_basket_truth(orders, tables["train"])
del tables
split_file = DERIVED / "split.csv"
if not split_file.exists():
    raise FileNotFoundError("Run notebook 01 first to create data/derived/split.csv")
assignment = pd.read_csv(split_file)
assert assignment["user_id"].is_unique
assert set(assignment["split"]) == {"train", "validation", "test"}
assert set(assignment["user_id"]) == set(truth), "The saved split must match the labelled users"
split = {part: assignment.loc[assignment["split"] == part, "user_id"].to_numpy()
         for part in ("train", "validation", "test")}
train_truth = {u: truth[u] for u in split["train"]}
val_truth = {u: truth[u] for u in split["validation"]}
print({part: len(users) for part, users in split.items()})
print(f"Loaded in {time.time() - t:.1f}s")

## 3. Candidates and features

Candidates are **all products previously bought by the user plus the 20 most popular
training-history products**. This keeps the problem smaller than scoring the entire
catalogue for every user and allows a limited number of new products. No negatives
are sampled. The candidate policy is fixed before examining validation targets.

| Features | Meaning |
|---|---|
| user_orders, user_mean_basket_size, user_unique_products | amount and variety of prior history |
| purchase_count, purchase_rate | number and fraction of prior orders containing this product |
| orders_since_last, order_span, in_last_order | recency and span between first and last purchases |
| product_orders, product_users, product_order_share | popularity among training users' prior orders |
| aisle_id, department_id | product categories, handled as categorical features |

Counts are zero for unseen products. Recency/span are missing, which LightGBM handles.
**Candidate recall** is the mean share of actual next-basket items present in the candidate
set: an upper bound on recall for rankings drawn from these candidates. Final recall
always uses the entire target basket, including products outside the candidate set.

In [ ]:
stats = product_features(orders, prior, split["train"])
fallback = stats["product_id"].head(K).tolist()
assert len(fallback) == K, "Need at least K distinct training-history products"
train_features = make_features(orders, prior, products, split["train"], stats, N_POPULAR)
validation_features = make_features(orders, prior, products, split["validation"], stats, N_POPULAR)


def candidate_summary(frame, targets):
    hits = pd.Series(candidate_labels(frame, targets), index=frame["user_id"].to_numpy()).groupby(level=0).sum()
    return {
        "candidates/user": len(frame) / len(targets),
        "candidate recall": np.mean([hits.get(u, 0) / len(actual) for u, actual in targets.items()]),
        "candidate hit-rate": np.mean([hits.get(u, 0) > 0 for u in targets]),
    }


print(f"Feature frames: {(train_features.memory_usage().sum() + validation_features.memory_usage().sum()) / 1e6:.1f} MB")
pd.DataFrame({"train": candidate_summary(train_features, train_truth),
              "validation": candidate_summary(validation_features, val_truth)}).T

## 4. Tune on validation

Fit binary purchase prediction with log loss. Early stopping chooses the number of
boosting rounds on validation log loss; the small tree-size comparison is selected by
**validation hit-rate@10**, breaking ties by precision@10. These are recommendation
metrics, rather than accuracy on the many negative user-product rows.

The selected model stays fitted on training users only. Test targets are used only in
section 5, after all settings are fixed. Predicted probabilities are used to rank candidates;
there is no classification threshold.

In [ ]:
rows, models = [], {}
for leaves in (15, 31):
    t = time.time()
    model = fit_lightgbm(train_features, train_truth, validation_features, val_truth,
                         num_leaves=leaves)
    recs = recommend_lightgbm(model, validation_features, split["validation"], K, fallback)
    models[leaves] = model
    rows.append({"num_leaves": leaves, "best_iteration": model.best_iteration_,
                 "seconds": time.time() - t, **evaluate_at_k(recs, val_truth, K)})
tuning = pd.DataFrame(rows).sort_values(["hit_rate@10", "precision@10"],
                                       ascending=False, kind="stable", ignore_index=True)
best = tuning.iloc[0]
model = models[int(best["num_leaves"])]
print(f"Selected {int(best['num_leaves'])} leaves, {model.best_iteration_} boosting rounds")
tuning.round(4)

In [ ]:
importance = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                       index=FEATURES).sort_values()
ax = importance.plot.barh(figsize=(8, 4), title="LightGBM feature importance (gain)")
ax.set_xlabel("Total training split gain")
plt.tight_layout()
plt.show()

## 5. Final test comparison (used once)

Settings and product statistics are now frozen. Test users are processed in batches to
limit feature-building memory. As in notebook 02, precision@10, recall@10 and hit-rate@10
are averaged over **all test users**. Products already bought may be recommended.

The popularity baseline below uses the same training-history fallback as LightGBM.
If notebook 02's saved predictions are available, evaluate those on the same users too.
Those rules were mined from all users' prior baskets, so their global history pool differs
from the training-user pool used for LightGBM's product statistics and popularity baseline.

In [ ]:
# Release the large training/validation frames before building test features.
del train_features, validation_features, models

test_users = split["test"]
test_truth = {u: truth[u] for u in test_users}
test_recs, coverage = {}, []
for start in range(0, len(test_users), 5_000):
    users = test_users[start:start + 5_000]
    frame = make_features(orders, prior, products, users, stats, N_POPULAR)
    test_recs.update(recommend_lightgbm(model, frame, users, K, fallback))
    summary = candidate_summary(frame, {u: test_truth[u] for u in users})
    coverage.append({"users": len(users), **summary})
    del frame
coverage = pd.DataFrame(coverage)
print("Test candidate coverage:")
print(coverage.drop(columns="users").mul(coverage["users"], axis=0).sum() / len(test_users))

results = [
    {"model": "Popularity (training history)", **evaluate_at_k({u: fallback for u in test_users}, test_truth, K)},
    {"model": "LightGBM", **evaluate_at_k(test_recs, test_truth, K)},
]
rule_file = DERIVED / "rule_recommendations_test.csv"
if rule_file.exists():
    saved = pd.read_csv(rule_file).sort_values(["user_id", "rank"])
    rule_recs = saved.groupby("user_id")["product_id"].apply(list).to_dict()
    assert set(test_users) <= set(rule_recs), "Saved rules do not cover the shared test users"
    results.append({"model": "Association rules (notebook 02)", **evaluate_at_k(rule_recs, test_truth, K)})
else:
    print("Run notebook 02 to include its saved rule recommendations in the comparison.")
results = pd.DataFrame(results)
results.round(4)

In [ ]:
# Same long format as rule_recommendations_test.csv.
assert set(test_recs) == set(test_users)
assert all(len(ps) == K and len(set(ps)) == K for ps in test_recs.values())
pd.DataFrame([(u, rank + 1, p) for u, ps in test_recs.items() for rank, p in enumerate(ps)],
             columns=["user_id", "rank", "product_id"]).to_csv(
                 DERIVED / "lightgbm_recommendations_test.csv", index=False)
results.to_csv(DERIVED / "lightgbm_test_metrics.csv", index=False)
tuning.to_csv(DERIVED / "lightgbm_validation.csv", index=False)
model.booster_.save_model(str(DERIVED / "lightgbm_model.txt"))
print("Saved recommendations, metrics, validation settings and the fitted model")

## Reading the results

- Compare LightGBM's test hit-rate@10 with both baselines; report precision and recall too.
- Candidate recall measures a separate limitation: no scoring model can recover a product
  excluded by candidate generation. This policy primarily models repeat purchases.
- Feature gain describes training splits, not a causal effect of a feature on purchases.
- Full-data scores and runtime should be recorded only after running this notebook with
  the original CSVs. The toy example checks counts and label separation, not model quality.

References: [LightGBM classifier](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html)
and [early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html).